# Analyse des données et Modélisation des prix des voitures

Ce notebook effectue une analyse exploratoire des données (EDA), un prétraitement des données, 
ainsi que l'entraînement et l'évaluation de plusieurs modèles de régression pour prédire le prix des voitures.

---


## Importation des bibliothèques

In [ ]:
# Importation des bibliothèques nécessaires
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import datetime

sns.set(rc={'figure.figsize': [10, 10]}, font_scale=1.2)

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score ,mean_squared_error


## Chargement des données et aperçu

In [ ]:
# Chargement des données
data = pd.read_csv(r"Datas\train.csv")

# Aperçu des premières lignes
data.head()

# Informations générales sur les données
data.info()


## Vérification des valeurs manquantes et dupliquées

In [ ]:
# Vérification des valeurs dupliquées et des valeurs manquantes
print("Valeurs dupliquées :", data.duplicated().sum())
print("Valeurs manquantes :")
print(data.isna().sum())

# Suppression de la colonne 'New_Price' car elle contient trop de valeurs manquantes
data.drop("New_Price", axis=1, inplace=True)


## Nettoyage des colonnes texte et conversion des types

In [ ]:
# Nettoyage des noms des voitures
data.Name = data.Name.apply(lambda x: " ".join(x.split()[0:2]))

# Suppression des valeurs manquantes dans certaines colonnes
data.dropna(subset=["Mileage", "Power", "Seats"], inplace=True)

# Conversion des colonnes numériques au bon format
data.Mileage = data.Mileage.apply(lambda x: x.split()[0]).astype("Float64")
data.Power = data.Power.apply(lambda x: x.split()[0]).astype("Float64")
data.Engine = data.Engine.apply(lambda x: x.split()[0]).astype("Int64")


## Analyse de la corrélation

In [ ]:
# Matrice de corrélation entre les variables numériques
colums = ["Mileage","Engine","Power","Price"]
first_corr_tabel = data[colums].corr()

sns.heatmap(first_corr_tabel, annot=True)
plt.show()


## Préparation des données pour la modélisation

In [ ]:
# Préparation des données pour la modélisation
x = data.drop("Price", axis=1)
y = data["Price"]

# Transformation des variables catégoriques
nominal = ["Name", "Transmission", "Location", "Fuel_Type", "Type"]
ordinal = ["Owner_Type"]
numerical = x.select_dtypes(["Int64", "float64"]).columns

# Normalisation des données numériques
x[numerical] = StandardScaler().fit_transform(x[numerical])

# Transformation des variables ordinales
transformation = {"First": 3, "Second": 2, "Third": 1, "Fourth & Above": 0}
x["Owner_Type"] = x["Owner_Type"].map(transformation)

# Encodage des variables nominales
import category_encoders as ce
binaryencoder = ce.BinaryEncoder(cols=nominal)
x = binaryencoder.fit_transform(x)


## Entraînement et évaluation des modèles

In [ ]:
# Division des données en ensembles d'entraînement et de test
x_train, x_test, y_train, y_test = train_test_split(x, y, random_state=42, shuffle=True, test_size=0.2)

# Entraînement de plusieurs modèles de régression
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree (max_depth=6)": DecisionTreeRegressor(max_depth=6),
    "Random Forest (n_estimators=100)": RandomForestRegressor(n_estimators=100, max_depth=6, max_leaf_nodes=42, max_features=50),
    "XGBoost (n_estimators=60)": XGBRegressor(n_estimators=60, max_depth=4)
}

# Évaluation des modèles
for name, model in models.items():
    model.fit(x_train, np.log(y_train))
    y_pred_train = model.predict(x_train)
    y_pred_test = model.predict(x_test)

    print(f"Modèle : {name}")
    print(f"R² sur l'entraînement : {r2_score(np.log(y_train), y_pred_train)}")
    print(f"R² sur le test : {r2_score(np.log(y_test), y_pred_test)}")
    print("-" * 50)
